# The Normal Equations
**Topic:** Statistics for Machine Learning

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import ipywidgets as widgets
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, clear_output
from scipy import stats
from scipy.linalg import lstsq
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** what the normal equations solve and why "normal" refers to perpendicularity, not the bell curve
- **Explain** the MSE loss surface geometrically: why the best-fit line minimizes the sum of squared vertical distances from the data points
- **Interpret** when sklearn's LinearRegression uses the normal equations versus gradient descent, and what the matrix inverse has to do with computational cost

> **Tip:** Move the slope and intercept sliders and watch how the MSE value changes on the surface. The bowl has exactly one lowest point — click "Solve" to jump there directly. Then ask: could you get to the same point by walking downhill step by step?

---
## How we got here

In 02: Information Theory Basics we saw that cross-entropy is the loss function for classification. For regression, the analogous loss is mean squared error: it measures the average squared vertical distance between the predicted line and the actual data points. The normal equations give the exact, closed-form solution to minimizing that MSE loss — no iteration required, no learning rate to choose. They are derived by setting the gradient of the loss to zero and solving the resulting matrix equation.

---
## Why this matters for data science

`sklearn.linear_model.LinearRegression` uses scipy's least-squares solver under the hood — which computes essentially the same solution as the normal equations — for datasets that fit in memory. For larger feature spaces or regularized models (Ridge, Lasso), sklearn switches to iterative methods. Understanding the normal equations tells you exactly what LinearRegression is computing, why it always gives the same answer (unlike gradient descent), and why it can become slow when you have thousands of features.

---
## Try it yourself

In [2]:
# Fixed dataset: 25 points from y = 2x + 1 + Gaussian noise
np.random.seed(17)
x_data = np.linspace(0, 5, 25)
y_data = 2.0 * x_data + 1.0 + np.random.normal(0, 1.2, 25)

# Normal equations solution
X_mat = np.column_stack([np.ones(len(x_data)), x_data])
beta_opt = np.linalg.lstsq(X_mat, y_data, rcond=None)[0]
intercept_opt, slope_opt = beta_opt[0], beta_opt[1]

# Precompute MSE surface
sl_range  = np.linspace(-1, 5, 60)
ic_range  = np.linspace(-3, 5, 60)
SL, IC = np.meshgrid(sl_range, ic_range)
MSE_surf = np.zeros_like(SL)
for i in range(SL.shape[0]):
    for j in range(SL.shape[1]):
        pred = IC[i,j] + SL[i,j] * x_data
        MSE_surf[i,j] = np.mean((y_data - pred)**2)

out = Output()

slope_s = FloatSlider(value=1.0, min=-1.0, max=5.0, step=0.1,
    description="Slope:", style={"description_width": "100px"}, layout=widgets.Layout(width="440px"))
intercept_s = FloatSlider(value=0.0, min=-3.0, max=5.0, step=0.1,
    description="Intercept:", style={"description_width": "100px"}, layout=widgets.Layout(width="440px"))
solve_btn = Button(description="Solve with Normal Equations",
    button_style="primary", layout=widgets.Layout(width="280px"))

def render(slope, intercept):
    pred        = intercept + slope * x_data
    mse_current = np.mean((y_data - pred)**2)
    mse_opt     = np.mean((y_data - (intercept_opt + slope_opt * x_data))**2)

    fig = make_subplots(rows=1, cols=2,
        specs=[[{"type": "surface"}, {"type": "scatter"}]],
        subplot_titles=[f"MSE Surface  (current = {mse_current:.3f})", "Fit vs Data"])

    fig.add_trace(go.Surface(x=SL, y=IC, z=MSE_surf, opacity=0.7,
        colorscale="Blues", showscale=False, name="MSE surface"), row=1, col=1)
    fig.add_trace(go.Scatter3d(x=[slope], y=[intercept], z=[mse_current],
        mode="markers", marker=dict(size=8, color=PALETTE["secondary"]),
        name="Current params"), row=1, col=1)
    fig.add_trace(go.Scatter3d(x=[slope_opt], y=[intercept_opt], z=[mse_opt],
        mode="markers", marker=dict(size=8, color=PALETTE["accent"], symbol="diamond"),
        name="Optimal (NE)"), row=1, col=1)

    fig.add_trace(go.Scatter(x=x_data, y=y_data, mode="markers",
        marker=dict(color=PALETTE["muted"], size=7), name="Data"), row=1, col=2)
    x_line = np.array([x_data.min(), x_data.max()])
    fig.add_trace(go.Scatter(x=x_line, y=intercept + slope * x_line, mode="lines",
        line=dict(color=PALETTE["secondary"], width=2.5, dash="dash"), name="Current line"), row=1, col=2)
    fig.add_trace(go.Scatter(x=x_line, y=intercept_opt + slope_opt * x_line, mode="lines",
        line=dict(color=PALETTE["accent"], width=2), name="Normal equations"), row=1, col=2)

    fig.update_layout(
        title=dict(
            text=f"β = (XᵀX)⁻¹Xᵀy  |  Optimal: slope = {slope_opt:.3f}, intercept = {intercept_opt:.3f}",
            font=dict(size=FONT["size_title"])),
        paper_bgcolor=PALETTE["background"],
        height=480,
        margin=dict(t=80, b=20, l=20, r=20),
        showlegend=True,
        scene=dict(xaxis_title="Slope", yaxis_title="Intercept", zaxis_title="MSE"),
    )
    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))
        print(f"  Current:  slope={slope:.2f}, intercept={intercept:.2f}, MSE={mse_current:.4f}")
        print(f"  Optimal:  slope={slope_opt:.4f}, intercept={intercept_opt:.4f}, MSE={mse_opt:.4f}")

def on_change(change):
    render(slope_s.value, intercept_s.value)

def on_solve(btn):
    slope_s.value     = round(slope_opt, 1)
    intercept_s.value = round(intercept_opt, 1)
    render(slope_opt, intercept_opt)

slope_s.observe(on_change, names="value")
intercept_s.observe(on_change, names="value")
solve_btn.on_click(on_solve)

display(VBox([slope_s, intercept_s, solve_btn, out]))
render(slope_s.value, intercept_s.value)

---
## What's happening?

Linear regression fits the model ŷ = Xβ where X is the data matrix (rows = samples, columns = features including a bias column of ones), β is the vector of coefficients, and ŷ is the vector of predictions.

The MSE loss is:

$$\text{MSE}(\beta) = \frac{1}{n}\|y - X\beta\|^2$$

To find the minimum, take the derivative with respect to β, set it to zero, and solve:

$$\frac{d}{d\beta}\text{MSE} = -\frac{2}{n}X^T(y - X\beta) = 0$$

$$X^TX\beta = X^Ty$$

$$\beta = (X^TX)^{-1}X^Ty$$

This is the normal equations solution. "Normal" here means perpendicular: the residuals (y − Xβ) are geometrically orthogonal to the column space of X. It is a fact about geometry, not Gaussian distributions.

| Method | How it finds the minimum | Pros | Cons | When sklearn uses it |
|--------|-------------------------|------|------|---------------------|
| Normal equations | Direct: β = (XᵀX)⁻¹Xᵀy | Exact, no iterations, no learning rate | O(p³) matrix inverse — slow for many features | Small/medium p (features) |
| scipy lstsq (SVD) | SVD-based least squares | Numerically stable, handles near-singular XᵀX | Slightly slower than direct inverse | sklearn `LinearRegression` default |
| Gradient descent | Iterative: β ← β − α∇MSE | Scales to millions of features | Approximate, learning rate sensitive | Large datasets, SGD variants |

---
## Real-world example: LinearRegression on Simulated Housing Data

The chart below demonstrates that sklearn's LinearRegression produces the same line as solving the normal equations directly using numpy. They are computing the same mathematical object by different numerical routes.

Notice:

- **Notice:** The sklearn fit and the numpy normal equations solution give identical coefficient values (to floating-point precision)
- **Notice:** Residuals — the vertical distances from points to the line — are minimized collectively, not individually
- **Notice:** The residuals are orthogonal to the fitted line in the matrix algebra sense — that is what "normal" means

In [4]:
np.random.seed(7)
n_house = 80
sqft = np.random.uniform(500, 3000, n_house)
price = 150 * sqft + 50_000 + np.random.normal(0, 30_000, n_house)

# sklearn LinearRegression
lr = LinearRegression()
lr.fit(sqft.reshape(-1,1), price)
sk_slope, sk_intercept = lr.coef_[0], lr.intercept_

# Direct normal equations
Xh = np.column_stack([np.ones(n_house), sqft])
beta_ne = np.linalg.lstsq(Xh, price, rcond=None)[0]
ne_intercept, ne_slope = beta_ne[0], beta_ne[1]

x_line = np.array([sqft.min(), sqft.max()])

fig = go.Figure()
fig.add_trace(go.Scatter(x=sqft, y=price, mode="markers",
    marker=dict(color=PALETTE["muted"], size=7, opacity=0.7), name="Homes"))
fig.add_trace(go.Scatter(x=x_line, y=sk_intercept + sk_slope * x_line,
    mode="lines", line=dict(color=PALETTE["primary"], width=3), name="sklearn LinearRegression"))
fig.add_trace(go.Scatter(x=x_line, y=ne_intercept + ne_slope * x_line,
    mode="lines", line=dict(color=PALETTE["secondary"], width=2, dash="dot"), name="Normal equations (numpy)"))

fig.update_layout(
    title=dict(text=f"sklearn vs Normal Equations  |  slope={sk_slope:.1f}, intercept={sk_intercept:,.0f}",
               font=dict(size=FONT["size_title"])),
    xaxis_title="Square footage",
    yaxis_title="Price ($)",
    paper_bgcolor=PALETTE["background"],
    plot_bgcolor=PALETTE["background"],
    height=420,
    margin=dict(t=60, b=40, l=60, r=20),
)
fig.update_xaxes(autorange=True)
fig.update_yaxes(autorange=True)
display(go.FigureWidget(fig))
print(f"  sklearn:          slope={sk_slope:.4f},  intercept={sk_intercept:,.2f}")
print(f"  Normal equations: slope={ne_slope:.4f},  intercept={ne_intercept:,.2f}")
print(f"  Max coefficient difference: {max(abs(sk_slope-ne_slope), abs(sk_intercept-ne_intercept)):.2e}")

FigureWidget({
    'data': [{'marker': {'color': '#868E96', 'opacity': 0.7, 'size': 7},
              'mode': 'markers',
              'name': 'Homes',
              'type': 'scatter',
              'uid': '9028ffb8-38d1-4b98-ab55-3cbf76a42282',
              'x': {'bdata': ('AlkMcSqWhUCOW9cNmCOjQFPH6qEX8J' ... 'RdO5VA5XiO/kJ8lEBXRSJlMIyTQA=='),
                    'dtype': 'f8'},
              'y': {'bdata': ('t9kMXw6nAUGhJU9W2FEXQVgkgYryZx' ... '5TGQ1BfVSxotIbDUH7TPRKyv4JQQ=='),
                    'dtype': 'f8'}},
             {'line': {'color': '#4C6EF5', 'width': 3},
              'mode': 'lines',
              'name': 'sklearn LinearRegression',
              'type': 'scatter',
              'uid': 'ad0f03fd-4581-442e-91d2-e2245fadd21c',
              'x': {'bdata': 'yE3IQRN5f0AKgkqT8gGnQA==', 'dtype': 'f8'},
              'y': {'bdata': 'mEP8uTobAEFpKhVbvK0dQQ==', 'dtype': 'f8'}},
             {'line': {'color': '#F76707', 'dash': 'dot', 'width': 2},
              'mode': 'lines

  sklearn:          slope=145.1261,  intercept=58,862.61
  Normal equations: slope=145.1261,  intercept=58,862.61
  Max coefficient difference: 8.73e-11


---
## Key takeaway

> **The normal equations β = (XᵀX)⁻¹Xᵀy give the exact minimum of the MSE loss in a single matrix operation — sklearn's LinearRegression uses this solution, and the matrix inverse is why it can be slow when you have thousands of features.**

---
*Next up: Regularization Mathematics — adding a penalty term to the loss that prevents coefficients from growing too large, trading a small increase in bias for a large reduction in variance.*